In [22]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [23]:
categories = ['comp.graphics','sci.space','misc.forsale']   # category subset
component_list = [50, 100, 200]

In [24]:
# load dataste

dataset = fetch_20newsgroups(
    subset='all',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)

documents = dataset.data
labels_true = dataset.target

print(f"Total documents loaded: {len(documents)}")

Total documents loaded: 2935


In [25]:
# TF-IDF vectorisation

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_df=0.5
)

X_tfidf = vectorizer.fit_transform(documents)

terms = vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", X_tfidf.shape)

TF-IDF matrix shape: (2935, 29846)


In [26]:
def print_top_terms(svd_model, terms, n_topics=5, n_words=10):

    topic_text = ""

    for i, comp in enumerate(svd_model.components_[:n_topics]):

        term_weights = zip(terms, comp)

        sorted_terms = sorted(
            term_weights,
            key=lambda x: abs(x[1]),
            reverse=True
        )[:n_words]

        topic_words = [t[0] for t in sorted_terms]

        topic_line = f"Topic {i+1}: {', '.join(topic_words)}"

        print(topic_line)

        topic_text += topic_line + "\n"

    return topic_text

In [27]:
results = []
all_topics_text = ""

for components in component_list:

    print(f"Running TruncatedSVD with {components} components")

    svd = TruncatedSVD(
        n_components=components,
        random_state=42
    )

    X_reduced = svd.fit_transform(X_tfidf)

    explained_variance = svd.explained_variance_ratio_.sum()

    print("Explained Variance Ratio:", explained_variance)


    # KMeans clustering
    kmeans = KMeans(
        n_clusters=len(categories),
        random_state=42,
        n_init=10
    )

    cluster_labels = kmeans.fit_predict(X_reduced)

    silhouette = silhouette_score(X_reduced, cluster_labels)

    print("Silhouette Score:", silhouette)


    results.append([
        components,
        explained_variance,
        silhouette
    ])

Running TruncatedSVD with 50 components
Explained Variance Ratio: 0.10262099697738207
Silhouette Score: 0.10237628502506392
Running TruncatedSVD with 100 components
Explained Variance Ratio: 0.16289622009649937
Silhouette Score: 0.07448185656742322
Running TruncatedSVD with 200 components
Explained Variance Ratio: 0.25656374961270817
Silhouette Score: 0.04691731431219422


In [28]:
#getting topics

with open("lsa_topic_terms.txt", "w") as f:
    f.write(all_topics_text)

print("\nSaved file: lsa_topic_terms.txt")


Saved file: lsa_topic_terms.txt


In [29]:
df_results = pd.DataFrame(
    results,
    columns=[
        "components",
        "explained_variance_ratio",
        "silhouette_score"
    ]
)

print("\nFinal Results Table:\n")
print(df_results)


Final Results Table:

   components  explained_variance_ratio  silhouette_score
0          50                  0.102621          0.102376
1         100                  0.162896          0.074482
2         200                  0.256564          0.046917


In [30]:
print("\nFinal Results Table:\n")
print(df_results)


Final Results Table:

   components  explained_variance_ratio  silhouette_score
0          50                  0.102621          0.102376
1         100                  0.162896          0.074482
2         200                  0.256564          0.046917


TruncatedSVD successfully reduced the dimensionality of the sparse TF-IDF matrix while preserving important semantic structure. As the number of components increased from 50 to 200, the explained variance ratio increased, indicating better information retention. However, clustering quality measured using silhouette score slightly decreased at higher dimensions due to reduced separation between clusters. This demonstrates the trade-off between dimensionality reduction and clustering performance, confirming that SVD effectively extracts latent topics from text datasets.